In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kaushalvaid/celegan-all-phases/celeba_32x32_gray.pt
/kaggle/input/datasets/kaushalvaid/celegan-all-phases/celeba_64x64_hr.pt
/kaggle/input/datasets/kaushalvaid/celegan-all-phases/celeba_16x16_lr.pt


### Imports

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
import torchvision.utils as vutils

import random
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

### Helper functions

In [3]:
def save_image_grid(fake_samples, epoch, path="outputs/"):
    """
        Saves a grid of generated images to disk.
        fake_samples: (B, 1, H, W) tensor in [-1, 1]
    """

    os.makedirs(path, exist_ok=True)

    # denormalise for display
    fake_samples = (fake_samples + 1) / 2

    grid = vutils.make_grid(fake_samples, nrow=8, padding=2, normalize=False)
    grid_np = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(grid_np)
    plt.axis("off")
    plt.title(f"Epoch {epoch}")
    plt.tight_layout()
    plt.savefig(os.path.join(path, f"epoch_{epoch:03d}.png"), dpi=100)
    plt.close()

def plot_loss_curve(loss_G, loss_D, path="outputs/"):
    """
        Saves G and D loss curves across epochs.
    """
    os.makedirs(path, exist_ok=True)

    plt.figure(figsize=(10, 4))
    plt.plot(loss_G, label="Generator Loss", color="steelblue")
    plt.plot(loss_D, label="Discriminator Loss", color="tomato")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("GAN Training Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(path, "loss_curve.png"), dpi=100)
    plt.close()

def load_and_display_progression(output_dir, epochs):
    """
        Notebook-only function.
        Loads saved grids from disk and plots them side by side for visual progression.
        epochs: e.g. [1, 5, 10, 25, 50]
    """
    fig, axes = plt.subplots(1, len(epochs), figsize=(4 * len(epochs), 4))

    for ax, epoch in zip(axes, epochs):
        img_path = os.path.join(output_dir, f"epoch_{epoch:03d}.png")
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.set_title(f"Epoch {epoch}")
        ax.axis("off")

    plt.suptitle("GAN Progression", fontsize=14)
    plt.tight_layout()
    plt.show()


### Model

In [4]:
# VGG feature extractor
class VGGFeatureExtractor(nn.Module):
    """
        Extracts features from VGG19 at relu3_3 (layer index 18).
        Kept it frozen, used only for perceptual loss computation
    """
    def __init__(self):
        super().__init__()

        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT)

        # take only upto relu3_3
        self.features = nn.Sequential(
            *list(vgg.features.children())[:18]
        )
        # freeze
        for param in self.features.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.features(x)
    

# residual block
class ResidualBlock(nn.Module):
    """
        Conv -> BN -> PReLU -> Conv -> BN -> skip conn
    """
    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(), # better than LeakyReLU for skip conns
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
    
    def forward(self, x):
        return x + self.block(x) # skip conn

# upsample block
class UpsampleBlock(nn.Module):
    """
        Conv -> PixelShuffle(r=2) -> PReLU
        Doubles spatial dims: (B, C, H, W) -> (B, C, H*2, W*2)
        Conv must output C * r^2 = C * 4 channels to work
    """
    def __init__(self, channels=64, upscale=2):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels * upscale ** 2, kernel_size=3, stride=1, padding=1),
            nn.PixelShuffle(upscale), # (B, C*4, H, W) -> (B, C, H*2, W*2)
            nn.PReLU()
        )

    def forward(self, x):
        return self.block(x)

# G
class Generator(nn.Module):
    """
        Input : LR image (B, 3, 16, 16)
        Output: HR image (B, 3, 64, 64)
        4x upscaling via 2x pixelshuffle blocks
    """

    def __init__(self, num_res_blocks=16, channels=64):
        super().__init__()

        # initial feat extraction from LR image
        self.initial = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=9, stride=1, padding=4), # 9x9 large kernel for large receptive field - output feat map preserves 
            # (N - K + 2P)/S + 1
            nn.PReLU()
        )

        # 16 res blocks
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(channels) for _ in range(num_res_blocks)]
        )

        # post residual conv + BN - before final skip
        self.post_res = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels)
        )

        # two upsample blocks: 16 -> 32 -> 64
        self.upsample = nn.Sequential(
            UpsampleBlock(channels, upscale=2),
            UpsampleBlock(channels, upscale=2)
        )

        # output : RGB
        self.final = nn.Conv2d(channels, 3, kernel_size=9, stride=1, padding=4)
        self.tanh = nn.Tanh()

    def forward(self, x):
        initial = self.initial(x) # (B, 64, 16, 16)
        res = self.res_blocks(initial) # (B, 64, 16, 16)
        post = self.post_res(res) # (B, 64, 16, 16)
        fused = initial + post # gloabl skip -- across ALL res block
        up = self.upsample(fused) # (B, 64, 64, 64)
        return self.tanh(self.final(up)) # (B, 3, 64, 64)


# D
class Discriminator(nn.Module):
    """
        VGG-style classifier/
        Input: (B, 3, 64, 64) HR image - real or G's output
        Output: (B, 1) - real/fake probability
    """

    def __init__(self, features_d=64):
        super().__init__()

        def block(in_c, out_c, stride):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.LeakyReLU(0.2, inplace=True)
            )

        self.net = nn.Sequential(
            # no BN on first layer
            nn.Conv2d(3, features_d, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            block(features_d, features_d, stride=2), # 64 -> 32
            block(features_d, features_d * 2, stride=1),
            block(features_d * 2, features_d * 2, stride=2), # 32 -> 16
            block(features_d * 2, features_d * 4, stride=1),
            block(features_d * 4, features_d * 4, stride=2), # 16 -> 8
            block(features_d * 4, features_d * 8, stride=1),
            block(features_d * 8, features_d * 8, stride=2), # 8 -> 4

            nn.AdaptiveAvgPool2d(1), # (B, 512, 1, 1)
            nn.Flatten(), # (B, 512)
            nn.Linear(features_d * 8, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


### Dataset

In [5]:
def get_srgan_tensor_loader(
    lr_pt_path,
    hr_pt_path,
    batch_size,
):
    lr_tensor = torch.load(lr_pt_path, weights_only=True)
    hr_tensor = torch.load(hr_pt_path, weights_only=True)

    assert lr_tensor.shape[0] == hr_tensor.shape[0], \
        f"LR/HR sample count mismatch: {lr_tensor.shape[0]} vs {hr_tensor.shape[0]}"

    dataset = TensorDataset(lr_tensor, hr_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

def get_tensor_loader(pt_path, batch_size):
    tensor = torch.load(pt_path, weights_only=True)
    dataset = TensorDataset(tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=True
    )

### Train

In [6]:
# hyperparameters
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 1e-4 # lower thaan other -- ResNet is sensitive
BATCH_SIZE = 128 # memory heavy (64 on single GPU, 128 on double GPU utilization)
NUM_EPOCHS = 50
SAVE_EPOCHS = [1, 5, 10, 25, 50]

LR_PT_PATH = "/kaggle/input/datasets/kaushalvaid/celegan-all-phases/celeba_16x16_lr.pt"
HR_PT_PATH = "/kaggle/input/datasets/kaushalvaid/celegan-all-phases/celeba_64x64_hr.pt"

# loss weights
LAMBDA_PERCEPTUAL = 1.0
LAMBDA_ADV = 1e-3
LAMBDA_MSE = 1e-2

def train():
    # data
    loader = get_srgan_tensor_loader(
        lr_pt_path=LR_PT_PATH,
        hr_pt_path=HR_PT_PATH,
        batch_size=BATCH_SIZE
    )

    # model
    G = Generator(num_res_blocks=16).to(DEVICE)
    D = Discriminator().to(DEVICE)
    vgg = VGGFeatureExtractor().to(DEVICE)

    G = nn.DataParallel(G)
    D = nn.DataParallel(D)
    
    vgg.eval()

    # loss functions
    criterion_adv = nn.BCELoss()
    criterion_mse = nn.MSELoss() # pixel loss
    criterion_per = nn.MSELoss() # perceptual loss - MSE on VGG feats

    # optims
    opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

    # fixed samples for visualization
    fixed_lr, fixed_hr = next(iter(loader))
    fixed_lr = fixed_lr[:16].to(DEVICE) # show 16 samples per grid
    fixed_hr = fixed_hr[:16]

    loss_G_history = []
    loss_D_history = []

    print("--------------- Starting training ---------------")

    for epoch in range(1, NUM_EPOCHS + 1):
        loss_G_epoch = 0.0
        loss_D_epoch = 0.0

        for batch_idx, (lr, hr) in enumerate(tqdm(loader, desc=f"Epoch [{epoch}/{NUM_EPOCHS}]", leave=True)):
            lr = lr.to(DEVICE)
            hr = hr.to(DEVICE)
            B = lr.size(0)

            real_labels = torch.ones(B, 1).to(DEVICE) * 0.9
            fake_labels = torch.zeros(B, 1).to(DEVICE)

            # train D
            fake_hr = G(lr).detach()

            loss_D_real = criterion_adv(D(hr), real_labels)
            loss_D_fake = criterion_adv(D(fake_hr), fake_labels)
            loss_D = loss_D_real + loss_D_fake

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()
    
            # training G twice per D -- D was dominating a lot...
            for _ in range(2):
                # train G
                fake_hr = G(lr)
    
                # adversarial loss: food D
                loss_adv = criterion_adv(D(fake_hr), real_labels)
                
                # perceptual: VGG feat similarity
                with torch.no_grad():
                    features_real = vgg(hr)
                features_fake = vgg(fake_hr)
                loss_perceptual = criterion_per(features_fake, features_real)
    
                # pixel: coarse structure
                loss_mse = criterion_mse(fake_hr, hr)
    
                # combined: G loss
                loss_G = (LAMBDA_PERCEPTUAL * loss_perceptual
                        + LAMBDA_ADV * loss_adv
                        + LAMBDA_MSE * loss_mse
                )
    
                opt_G.zero_grad()
                loss_G.backward()
                opt_G.step()

            loss_G_epoch += loss_G.item()
            loss_D_epoch += loss_D.item()

        # end of epoch
        avg_G = loss_G_epoch / len(loader)
        avg_D = loss_D_epoch / len(loader)
        loss_G_history.append(avg_G)
        loss_D_history.append(avg_D)

        print(f"Epoch [{epoch}/{NUM_EPOCHS}] | loss_D: {avg_D:.4f} | loss_G: {avg_G:.4f}")

        if epoch in SAVE_EPOCHS:
            with torch.no_grad():
                fake_hr = G(fixed_lr)
                # save LR, fake HR, real HR side by side for comparison
                save_image_grid(fixed_lr,  epoch, path="outputs/lr/")
                save_image_grid(fake_hr,   epoch, path="outputs/fake_hr/")
                save_image_grid(fixed_hr[:16].to(DEVICE), epoch, path="outputs/real_hr/")

    plot_loss_curve(loss_G_history, loss_D_history, path="outputs/")


# calling train
train()


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


  0%|          | 0.00/548M [00:00<?, ?B/s]

  3%|▎         | 14.9M/548M [00:00<00:03, 155MB/s]

  6%|▌         | 33.2M/548M [00:00<00:03, 177MB/s]

 10%|▉         | 53.5M/548M [00:00<00:02, 193MB/s]

...

Epoch [22/50]:  30%|██▉       | 116/390 [01:26<03:23,  1.35it/s]

Epoch [22/50]:  30%|███       | 117/390 [01:26<03:22,  1.35it/s]

Epoch [22/50]:  30%|███       | 118/390 [01:38<18:09,  4.01s/it]

Epoch [22/50]:  31%|███       | 119/390 [02:22<1:12:37, 16.08s/it]

Epoch [22/50]:  31%|███       | 120/390 [03:27<2:17:28, 30.55s/it]

In [ ]:
load_and_display_progression(output_dir='/kaggle/working/outputs/lr', epochs=[1, 5, 10, 25, 50])

In [ ]:
load_and_display_progression(output_dir='/kaggle/working/outputs/fake_hr', epochs=[1, 5, 10, 25, 50])

In [ ]:
load_and_display_progression(output_dir='/kaggle/working/outputs/real_hr', epochs=[1, 5, 10, 25, 50])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('/kaggle/working/outputs/loss_curve.png')
plt.imshow(img)
plt.axis('off') # Optional: hides grid lines and pixel counts
plt.show()